# 衣類キーポイント検出 — HRNet-w48 学習

MMPose + HRNet-w48 を使って衣類のキーポイント検出モデルをfine-tuneする。

## セットアップ
1. ランタイム → ランタイムのタイプを変更 → GPU (T4) を選択
2. Google Driveにデータをアップロード済みであること

## 1. 環境構築

In [ ]:
# GPU確認
!nvidia-smi

In [ ]:
# MMPose + 依存ライブラリのインストール
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q -U openmim
!mim install -q mmengine mmcv mmdet mmpose

# 確認
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
import mmpose
print(f'MMPose: {mmpose.__version__}')

## 2. データ準備

In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

# プロジェクトディレクトリ（Drive上のパスに合わせて変更）
PROJECT_DIR = '/content/drive/MyDrive/keypoint-detection'
!ls {PROJECT_DIR}

In [ ]:
# ローカルにコピー（Driveからの直接読み込みは遅い）
!cp -r {PROJECT_DIR}/data /content/data
!cp -r {PROJECT_DIR}/configs /content/configs

# シンボリックリンク解決（実ファイルをコピー）
import os, shutil
for split in ['train', 'val']:
    img_dir = f'/content/data/images/{split}'
    for f in os.listdir(img_dir):
        path = os.path.join(img_dir, f)
        if os.path.islink(path):
            real = os.path.realpath(path)
            os.remove(path)
            # Driveから実画像をコピー
            src = f'{PROJECT_DIR}/data/images/{split}/{f}'
            if os.path.exists(src):
                shutil.copy2(src, path)
                print(f'Copied: {path}')

print('\nデータ準備完了')
!ls -la /content/data/images/train/
!ls -la /content/data/annotations/

## 3. 設定の確認

In [ ]:
# 学習対象のカテゴリを選択
CATEGORY = 'tops'  # 'tops', 'pants', 'skirt', 'dress', 'salopette'

CONFIG_FILE = f'/content/configs/{CATEGORY}_hrnet_w48.py'
WORK_DIR = f'/content/work_dirs/{CATEGORY}_hrnet_w48'

print(f'カテゴリ: {CATEGORY}')
print(f'設定ファイル: {CONFIG_FILE}')
print(f'出力先: {WORK_DIR}')

In [ ]:
# 設定ファイルのdata_rootを修正（Colab用）
from mmengine.config import Config

cfg = Config.fromfile(CONFIG_FILE)
cfg.data_root = '/content/data/'
cfg.train_dataloader.dataset.data_root = '/content/data/'
cfg.val_dataloader.dataset.data_root = '/content/data/'
cfg.val_evaluator.ann_file = f'/content/data/annotations/dummy_{CATEGORY}_val.json'
cfg.work_dir = WORK_DIR

# ダミーデータが少ないので学習パラメータを調整
cfg.train_cfg.max_epochs = 20  # ダミーテスト用に短縮
cfg.train_cfg.val_interval = 5
cfg.train_dataloader.batch_size = 2
cfg.val_dataloader.batch_size = 2
cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.num_workers = 2
cfg.train_dataloader.persistent_workers = False
cfg.val_dataloader.persistent_workers = False

# 修正した設定を保存
cfg.dump(f'/content/{CATEGORY}_config_colab.py')
print('設定ファイル保存完了')

## 4. 学習実行

In [ ]:
# 学習開始
!python -m mmpose.tools.train /content/{CATEGORY}_config_colab.py \
    --work-dir {WORK_DIR}

## 5. 評価

In [ ]:
import glob

# 最良チェックポイントを探す
best_ckpt = glob.glob(f'{WORK_DIR}/best_*.pth')
if best_ckpt:
    CKPT = best_ckpt[0]
else:
    # 最新のエポックを使用
    ckpts = sorted(glob.glob(f'{WORK_DIR}/epoch_*.pth'))
    CKPT = ckpts[-1] if ckpts else None

print(f'チェックポイント: {CKPT}')

In [ ]:
# 評価実行
if CKPT:
    !python -m mmpose.tools.test /content/{CATEGORY}_config_colab.py {CKPT}

## 6. 推論テスト（1枚の画像で確認）

In [ ]:
from mmpose.apis import init_model, inference_topdown
from mmpose.utils import register_all_modules
import cv2
import numpy as np
from matplotlib import pyplot as plt

register_all_modules()

if CKPT:
    model = init_model(f'/content/{CATEGORY}_config_colab.py', CKPT, device='cuda:0')

    # テスト画像
    test_images = {
        'tops': '/content/data/images/val/IMG_1722.jpg',
        'pants': '/content/data/images/val/IMG_1721.jpg',
        'skirt': '/content/data/images/val/IMG_1723.jpg',
        'dress': '/content/data/images/val/IMG_1724.jpg',
        'salopette': '/content/data/images/val/IMG_1722.jpg',  # ダミー流用
    }
    test_img = test_images.get(CATEGORY, list(test_images.values())[0])

    img = cv2.imread(test_img)
    h, w = img.shape[:2]
    bboxes = [[0, 0, w, h]]  # 画像全体

    results = inference_topdown(model, test_img, bboxes)

    if results:
        kps = results[0].pred_instances.keypoints[0]
        scores = results[0].pred_instances.keypoint_scores[0]

        # キーポイント名を取得
        from configs.categories import CATEGORY_MAP
        kp_names = CATEGORY_MAP[CATEGORY]["keypoints"]

        # 描画
        vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if max(w, h) > 1024:
            r = 1024 / max(w, h)
            vis = cv2.resize(vis, (int(w*r), int(h*r)))
            kps = kps * r

        for i, (pt, sc) in enumerate(zip(kps, scores)):
            x, y = int(pt[0]), int(pt[1])
            color = (0, 255, 0) if sc > 0.5 else (255, 0, 0)
            cv2.circle(vis, (x, y), 8, color, -1)
            label = kp_names[i] if i < len(kp_names) else str(i)
            cv2.putText(vis, f'{label}:{sc:.2f}', (x+10, y),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        plt.figure(figsize=(12, 12))
        plt.imshow(vis)
        plt.title(f'{CATEGORY} keypoint detection ({len(kps)} points)')
        plt.axis('off')
        plt.show()
    else:
        print('検出結果なし')

## 7. モデルをDriveに保存

In [ ]:
import shutil

if CKPT:
    # 最良モデルをDriveに保存
    save_dir = f'{PROJECT_DIR}/trained_models/{CATEGORY}'
    os.makedirs(save_dir, exist_ok=True)
    shutil.copy2(CKPT, f'{save_dir}/best.pth')
    shutil.copy2(f'/content/{CATEGORY}_config_colab.py', f'{save_dir}/config.py')
    print(f'保存先: {save_dir}')
    !ls -la {save_dir}

## 8. 全カテゴリ一括学習（本番用）

全カテゴリを順番に学習する場合は以下を実行:

In [ ]:
# 全カテゴリ一括学習
ALL_CATEGORIES = ['tops', 'pants', 'skirt', 'dress', 'salopette']

for cat in ALL_CATEGORIES:
    print(f'\n{"="*60}')
    print(f'学習開始: {cat}')
    print(f'{"="*60}\n')

    config_path = f'/content/configs/{cat}_hrnet_w48.py'
    work_dir = f'/content/work_dirs/{cat}_hrnet_w48'

    # 設定調整
    cfg = Config.fromfile(config_path)
    cfg.data_root = '/content/data/'
    cfg.train_dataloader.dataset.data_root = '/content/data/'
    cfg.val_dataloader.dataset.data_root = '/content/data/'
    cfg.val_evaluator.ann_file = f'/content/data/annotations/dummy_{cat}_val.json'
    cfg.work_dir = work_dir

    # 本番設定（データが揃ったら以下のコメントを外す）
    # cfg.train_dataloader.dataset.ann_file = f'annotations/{cat}_train.json'
    # cfg.val_dataloader.dataset.ann_file = f'annotations/{cat}_val.json'
    # cfg.val_evaluator.ann_file = f'/content/data/annotations/{cat}_val.json'
    # cfg.train_cfg.max_epochs = 210

    # ダミー用に短縮
    cfg.train_cfg.max_epochs = 10
    cfg.train_cfg.val_interval = 5
    cfg.train_dataloader.batch_size = 2
    cfg.val_dataloader.batch_size = 2
    cfg.train_dataloader.num_workers = 2
    cfg.val_dataloader.num_workers = 2
    cfg.train_dataloader.persistent_workers = False
    cfg.val_dataloader.persistent_workers = False

    colab_config = f'/content/{cat}_config_colab.py'
    cfg.dump(colab_config)

    !python -m mmpose.tools.train {colab_config} --work-dir {work_dir}

    # Driveに保存
    best = glob.glob(f'{work_dir}/best_*.pth')
    if best:
        save_dir = f'{PROJECT_DIR}/trained_models/{cat}'
        os.makedirs(save_dir, exist_ok=True)
        shutil.copy2(best[0], f'{save_dir}/best.pth')
        shutil.copy2(colab_config, f'{save_dir}/config.py')
        print(f'\n保存: {save_dir}')

print('\n全カテゴリ学習完了！')